In [0]:
spark.sql("USE CATALOG supply_chain_dev")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS lp_analytics
    COMMENT '3PL Unified Analytics Lakehouse — SAP + Salesforce + TMS'
""")

spark.sql("USE SCHEMA lp_analytics")

print("Catalog : supply_chain_dev")
print("Schema  : lp_analytics")
print("Ready!")

Catalog : supply_chain_dev
Schema  : lp_analytics
Ready!


In [0]:
spark.sql("""
    CREATE VOLUME IF NOT EXISTS supply_chain_dev.lp_analytics.raw_sources
    COMMENT 'Landing zone for SAP, Salesforce and TMS source files'
""")

import os

BASE_PATH = "/Volumes/supply_chain_dev/lp_analytics/raw_sources"

os.makedirs(f"{BASE_PATH}/sap/orders", exist_ok=True)
os.makedirs(f"{BASE_PATH}/sap/invoices", exist_ok=True)
os.makedirs(f"{BASE_PATH}/salesforce/accounts", exist_ok=True)
os.makedirs(f"{BASE_PATH}/salesforce/contracts", exist_ok=True)
os.makedirs(f"{BASE_PATH}/tms/shipments", exist_ok=True)
os.makedirs(f"{BASE_PATH}/checkpoints/sap", exist_ok=True)
os.makedirs(f"{BASE_PATH}/checkpoints/salesforce", exist_ok=True)
os.makedirs(f"{BASE_PATH}/checkpoints/tms", exist_ok=True)

print("Volume created: supply_chain_dev.lp_analytics.raw_sources")
print("Directories ready:")
print(f"  {BASE_PATH}/sap/orders")
print(f"  {BASE_PATH}/sap/invoices")
print(f"  {BASE_PATH}/salesforce/accounts")
print(f"  {BASE_PATH}/salesforce/contracts")
print(f"  {BASE_PATH}/tms/shipments")

Volume created: supply_chain_dev.lp_analytics.raw_sources
Directories ready:
  /Volumes/supply_chain_dev/lp_analytics/raw_sources/sap/orders
  /Volumes/supply_chain_dev/lp_analytics/raw_sources/sap/invoices
  /Volumes/supply_chain_dev/lp_analytics/raw_sources/salesforce/accounts
  /Volumes/supply_chain_dev/lp_analytics/raw_sources/salesforce/contracts
  /Volumes/supply_chain_dev/lp_analytics/raw_sources/tms/shipments


In [0]:
%pip install faker
dbutils.library.restartPython()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 40.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import csv
import random
import os
from datetime import datetime, timedelta
from faker import Faker
import uuid

fake = Faker()
random.seed(42)

BASE_PATH = "/Volumes/supply_chain_dev/lp_analytics/raw_sources"

# ── Master reference data ──────────────────────────────────────────────
CUSTOMERS = [
    {"id": "CUST001", "name": "Amazon Logistics", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST002", "name": "Walmart Supply Chain", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST003", "name": "Target Distribution", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST004", "name": "Home Depot Freight", "segment": "Mid-Market", "region": "North America"},
    {"id": "CUST005", "name": "Nike Global Logistics", "segment": "Enterprise", "region": "Global"},
    {"id": "CUST006", "name": "Apple Supply Chain", "segment": "Enterprise", "region": "Global"},
    {"id": "CUST007", "name": "Samsung Americas", "segment": "Mid-Market", "region": "North America"},
    {"id": "CUST008", "name": "IKEA Distribution", "segment": "Mid-Market", "region": "Global"},
    {"id": "CUST009", "name": "Costco Wholesale", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST010", "name": "Best Buy Logistics", "segment": "Mid-Market", "region": "North America"},
    {"id": "CUST011", "name": "Zara Supply Chain", "segment": "Mid-Market", "region": "Global"},
    {"id": "CUST012", "name": "Tesla Parts Logistics", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST013", "name": "Procter & Gamble", "segment": "Enterprise", "region": "Global"},
    {"id": "CUST014", "name": "Unilever Logistics", "segment": "Mid-Market", "region": "Global"},
    {"id": "CUST015", "name": "Johnson Controls", "segment": "SMB", "region": "North America"}
]

CARRIERS = ["CMA CGM", "Hapag-Lloyd", "MSC", "Evergreen", "COSCO", "FedEx Freight", "UPS Supply Chain", "DHL Global"]
TRADE_LANES = ["USLAX-CNSHA", "CNSHA-NLRTM", "DEHAM-USLAX", "SGSIN-USLAX", "KRPUS-NLRTM", "AEJEA-USLAX"]
SERVICE_TYPES = ["FCL", "LCL", "Air Freight", "Cross Border Truck"]
SALES_REPS = ["Sarah Mitchell", "James Okonkwo", "Priya Sharma", "David Chen", "Maria Rodriguez"]
INCOTERMS = ["FOB", "CIF", "EXW", "DAP", "DDP"]

print("Reference data defined!")
print(f"Customers : {len(CUSTOMERS)}")
print(f"Carriers  : {len(CARRIERS)}")
print(f"Trade lanes: {len(TRADE_LANES)}")
print(f"Sales reps : {len(SALES_REPS)}")


Reference data defined!
Customers : 15
Carriers  : 8
Trade lanes: 6
Sales reps : 5


In [0]:
import csv
import random
import os
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
random.seed(42)

BASE_PATH = "/Volumes/supply_chain_dev/lp_analytics/raw_sources"

CUSTOMERS = [
    {"id": "CUST001", "name": "Amazon Logistics", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST002", "name": "Walmart Supply Chain", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST003", "name": "Target Distribution", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST004", "name": "Home Depot Freight", "segment": "Mid-Market", "region": "North America"},
    {"id": "CUST005", "name": "Nike Global Logistics", "segment": "Enterprise", "region": "Global"},
    {"id": "CUST006", "name": "Apple Supply Chain", "segment": "Enterprise", "region": "Global"},
    {"id": "CUST007", "name": "Samsung Americas", "segment": "Mid-Market", "region": "North America"},
    {"id": "CUST008", "name": "IKEA Distribution", "segment": "Mid-Market", "region": "Global"},
    {"id": "CUST009", "name": "Costco Wholesale", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST010", "name": "Best Buy Logistics", "segment": "Mid-Market", "region": "North America"},
    {"id": "CUST011", "name": "Zara Supply Chain", "segment": "Mid-Market", "region": "Global"},
    {"id": "CUST012", "name": "Tesla Parts Logistics", "segment": "Enterprise", "region": "North America"},
    {"id": "CUST013", "name": "Procter & Gamble", "segment": "Enterprise", "region": "Global"},
    {"id": "CUST014", "name": "Unilever Logistics", "segment": "Mid-Market", "region": "Global"},
    {"id": "CUST015", "name": "Johnson Controls", "segment": "SMB", "region": "North America"}
]

CARRIERS = ["CMA CGM", "Hapag-Lloyd", "MSC", "Evergreen", "COSCO", "FedEx Freight", "UPS Supply Chain", "DHL Global"]
TRADE_LANES = ["USLAX-CNSHA", "CNSHA-NLRTM", "DEHAM-USLAX", "SGSIN-USLAX", "KRPUS-NLRTM", "AEJEA-USLAX"]
SERVICE_TYPES = ["FCL", "LCL", "Air Freight", "Cross Border Truck"]
SALES_REPS = ["Sarah Mitchell", "James Okonkwo", "Priya Sharma", "David Chen", "Maria Rodriguez"]
INCOTERMS = ["FOB", "CIF", "EXW", "DAP", "DDP"]

sap_orders = []
base_date = datetime(2024, 1, 1)

for i in range(300):
    customer = random.choice(CUSTOMERS)
    order_date = base_date + timedelta(days=random.randint(0, 365))
    service = random.choice(SERVICE_TYPES)

    if service == "FCL":
        freight_cost = round(random.uniform(2500, 8000), 2)
        revenue = round(freight_cost * random.uniform(1.15, 1.45), 2)
    elif service == "LCL":
        freight_cost = round(random.uniform(800, 3000), 2)
        revenue = round(freight_cost * random.uniform(1.20, 1.50), 2)
    elif service == "Air Freight":
        freight_cost = round(random.uniform(5000, 25000), 2)
        revenue = round(freight_cost * random.uniform(1.10, 1.35), 2)
    else:
        freight_cost = round(random.uniform(1200, 5000), 2)
        revenue = round(freight_cost * random.uniform(1.18, 1.40), 2)

    sap_orders.append({
        "sap_order_id": f"ORD{str(i+1).zfill(5)}",
        "customer_id": customer["id"],
        "customer_name": customer["name"],
        "order_date": order_date.strftime("%Y-%m-%d"),
        "service_type": service,
        "trade_lane": random.choice(TRADE_LANES),
        "incoterms": random.choice(INCOTERMS),
        "freight_cost_usd": freight_cost,
        "revenue_usd": revenue,
        "margin_usd": round(revenue - freight_cost, 2),
        "margin_pct": round(((revenue - freight_cost) / revenue) * 100, 2),
        "weight_kg": round(random.uniform(500, 30000), 2),
        "volume_cbm": round(random.uniform(5, 200), 2),
        "sap_division": random.choice(["Ocean Freight", "Air Freight", "Ground"]),
        "payment_terms": random.choice(["NET30", "NET45", "NET60"]),
        "order_status": random.choice(["CONFIRMED", "IN_PROGRESS", "INVOICED", "CLOSED"]),
        "created_by": random.choice(SALES_REPS),
        "source_system": "SAP_ERP",
        "extract_date": datetime.utcnow().strftime("%Y-%m-%d")
    })

orders_file = f"{BASE_PATH}/sap/orders/sap_orders_extract.csv"
with open(orders_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=sap_orders[0].keys())
    writer.writeheader()
    writer.writerows(sap_orders)

print(f"SAP Orders: {len(sap_orders)} records written")
print(f"Location  : {orders_file}")

/home/spark-b3ef089f-5960-4aa0-8bf9-b0/.ipykernel/306/command-4803637894559646-3113952886:76: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "extract_date": datetime.utcnow().strftime("%Y-%m-%d")


SAP Orders: 300 records written
Location  : /Volumes/supply_chain_dev/lp_analytics/raw_sources/sap/orders/sap_orders_extract.csv


In [0]:
sap_invoices = []
for order in random.sample(sap_orders, 250):
    invoice_date = datetime.strptime(order["order_date"], "%Y-%m-%d") + timedelta(days=random.randint(7, 30))
    sap_invoices.append({
        "invoice_id": f"INV{str(random.randint(100000, 999999))}",
        "sap_order_id": order["sap_order_id"],
        "customer_id": order["customer_id"],
        "invoice_date": invoice_date.strftime("%Y-%m-%d"),
        "invoice_amount_usd": order["revenue_usd"],
        "payment_status": random.choice(["PAID", "PENDING", "OVERDUE"]),
        "days_to_pay": random.randint(5, 75) if random.random() > 0.3 else None,
        "source_system": "SAP_ERP",
        "extract_date": datetime.now().strftime("%Y-%m-%d")
    })

invoices_file = f"{BASE_PATH}/sap/invoices/sap_invoices_extract.csv"
with open(invoices_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=sap_invoices[0].keys())
    writer.writeheader()
    writer.writerows(sap_invoices)

print(f"SAP Invoices: {len(sap_invoices)} records written")
print(f"Location    : {invoices_file}")

SAP Invoices: 250 records written
Location    : /Volumes/supply_chain_dev/lp_analytics/raw_sources/sap/invoices/sap_invoices_extract.csv


In [0]:
import json

# ── SALESFORCE ACCOUNTS ─────────────────────────────────────────────────
# Salesforce exports accounts as JSON — one record per customer

sf_accounts = []
for customer in CUSTOMERS:
    created_date = base_date + timedelta(days=random.randint(-365, 0))
    sf_accounts.append({
        "sf_account_id": f"SF{customer['id']}",
        "customer_id": customer["id"],
        "account_name": customer["name"],
        "segment": customer["segment"],
        "region": customer["region"],
        "account_owner": random.choice(SALES_REPS),
        "industry": random.choice(["Retail", "Manufacturing", "Technology", "Consumer Goods"]),
        "annual_revenue_usd": round(random.uniform(5000000, 500000000), 2),
        "employee_count": random.randint(500, 50000),
        "account_status": random.choice(["Active", "Active", "Active", "At Risk", "Churned"]),
        "nps_score": random.randint(20, 90),
        "created_date": created_date.strftime("%Y-%m-%d"),
        "last_activity_date": (datetime.now() - timedelta(days=random.randint(1, 60))).strftime("%Y-%m-%d"),
        "source_system": "SALESFORCE",
        "extract_date": datetime.now().strftime("%Y-%m-%d")
    })

accounts_file = f"{BASE_PATH}/salesforce/accounts/sf_accounts_extract.json"
with open(accounts_file, 'w') as f:
    for record in sf_accounts:
        f.write(json.dumps(record) + '\n')

print(f"Salesforce Accounts: {len(sf_accounts)} records written")

# ── SALESFORCE CONTRACTS ─────────────────────────────────────────────────
sf_contracts = []
for customer in CUSTOMERS:
    num_contracts = random.randint(1, 3)
    for c in range(num_contracts):
        start_date = base_date + timedelta(days=random.randint(-180, 180))
        sf_contracts.append({
            "contract_id": f"CON{str(random.randint(10000, 99999))}",
            "customer_id": customer["id"],
            "sf_account_id": f"SF{customer['id']}",
            "contract_type": random.choice(["Annual", "Multi-Year", "Spot"]),
            "service_scope": random.choice(SERVICE_TYPES),
            "contract_value_usd": round(random.uniform(50000, 5000000), 2),
            "start_date": start_date.strftime("%Y-%m-%d"),
            "end_date": (start_date + timedelta(days=random.randint(180, 730))).strftime("%Y-%m-%d"),
            "contract_status": random.choice(["Active", "Active", "Expired", "Pending"]),
            "renewal_probability_pct": random.randint(30, 95),
            "assigned_rep": random.choice(SALES_REPS),
            "source_system": "SALESFORCE",
            "extract_date": datetime.now().strftime("%Y-%m-%d")
        })

contracts_file = f"{BASE_PATH}/salesforce/contracts/sf_contracts_extract.json"
with open(contracts_file, 'w') as f:
    for record in sf_contracts:
        f.write(json.dumps(record) + '\n')

print(f"Salesforce Contracts: {len(sf_contracts)} records written")
print(f"Location: {BASE_PATH}/salesforce/")

Salesforce Accounts: 15 records written
Salesforce Contracts: 29 records written
Location: /Volumes/supply_chain_dev/lp_analytics/raw_sources/salesforce/


In [0]:
# ── TMS / CARGOWISE SHIPMENTS ───────────────────────────────────────────
# CargoWise exposes shipment data via REST API
# In production you'd call the API — here we simulate the JSON response

tms_shipments = []

for i in range(400):
    customer = random.choice(CUSTOMERS)
    order_ref = f"ORD{str(random.randint(1, 300)).zfill(5)}"
    ship_date = base_date + timedelta(days=random.randint(0, 365))
    
    # Simulate realistic transit times by service type
    service = random.choice(SERVICE_TYPES)
    if service == "FCL":
        transit_days = random.randint(14, 45)
    elif service == "LCL":
        transit_days = random.randint(18, 50)
    elif service == "Air Freight":
        transit_days = random.randint(2, 7)
    else:
        transit_days = random.randint(3, 12)

    eta = ship_date + timedelta(days=transit_days)
    
    # 82% on-time delivery rate — realistic for a 3PL
    on_time = random.random() < 0.82
    actual_arrival = eta if on_time else eta + timedelta(days=random.randint(1, 15))
    delay_days = (actual_arrival - eta).days

    tms_shipments.append({
        "shipment_id": f"SHP{str(i+1).zfill(6)}",
        "sap_order_ref": order_ref,
        "customer_id": customer["id"],
        "carrier": random.choice(CARRIERS),
        "service_type": service,
        "trade_lane": random.choice(TRADE_LANES),
        "origin_port": random.choice(TRADE_LANES).split("-")[0],
        "destination_port": random.choice(TRADE_LANES).split("-")[1],
        "ship_date": ship_date.strftime("%Y-%m-%d"),
        "eta": eta.strftime("%Y-%m-%d"),
        "actual_arrival": actual_arrival.strftime("%Y-%m-%d"),
        "transit_days_actual": transit_days + delay_days,
        "delay_days": delay_days,
        "on_time_delivery": on_time,
        "shipment_status": random.choice(["DELIVERED", "DELIVERED", "IN_TRANSIT", "CUSTOMS_HOLD"]),
        "container_count": random.randint(1, 20),
        "weight_kg": round(random.uniform(500, 30000), 2),
        "freight_charges_usd": round(random.uniform(800, 25000), 2),
        "customs_cleared": random.choice([True, True, True, False]),
        "source_system": "CARGOWISE_TMS",
        "extract_date": datetime.now().strftime("%Y-%m-%d")
    })

tms_file = f"{BASE_PATH}/tms/shipments/tms_shipments_extract.json"
with open(tms_file, 'w') as f:
    for record in tms_shipments:
        f.write(json.dumps(record) + '\n')

print(f"TMS Shipments: {len(tms_shipments)} records written")
print(f"Location     : {tms_file}")
print(f"\nData summary:")
print(f"  On-time deliveries: {sum(1 for s in tms_shipments if s['on_time_delivery'])}")
print(f"  Delayed shipments : {sum(1 for s in tms_shipments if not s['on_time_delivery'])}")
print(f"  Avg delay days    : {round(sum(s['delay_days'] for s in tms_shipments) / len(tms_shipments), 1)}")

TMS Shipments: 400 records written
Location     : /Volumes/supply_chain_dev/lp_analytics/raw_sources/tms/shipments/tms_shipments_extract.json

Data summary:
  On-time deliveries: 305
  Delayed shipments : 95
  Avg delay days    : 2.0


In [0]:
import os

sources = [
    f"{BASE_PATH}/sap/orders",
    f"{BASE_PATH}/sap/invoices",
    f"{BASE_PATH}/salesforce/accounts",
    f"{BASE_PATH}/salesforce/contracts",
    f"{BASE_PATH}/tms/shipments"
]

print("=== Source File Inventory ===\n")
total_size = 0
for folder in sources:
    files = os.listdir(folder)
    for file in files:
        filepath = os.path.join(folder, file)
        size_kb = round(os.path.getsize(filepath) / 1024, 1)
        total_size += size_kb
        print(f"  {file}")
        print(f"    Path : {filepath}")
        print(f"    Size : {size_kb} KB")
        print()

print(f"Total data size: {round(total_size, 1)} KB")
print("\nStep 1 Complete — all source data ready!")

=== Source File Inventory ===

  sap_orders_extract.csv
    Path : /Volumes/supply_chain_dev/lp_analytics/raw_sources/sap/orders/sap_orders_extract.csv
    Size : 52.2 KB

  sap_invoices_extract.csv
    Path : /Volumes/supply_chain_dev/lp_analytics/raw_sources/sap/invoices/sap_invoices_extract.csv
    Size : 18.6 KB

  sf_accounts_extract.json
    Path : /Volumes/supply_chain_dev/lp_analytics/raw_sources/salesforce/accounts/sf_accounts_extract.json
    Size : 6.4 KB

  sf_contracts_extract.json
    Path : /Volumes/supply_chain_dev/lp_analytics/raw_sources/salesforce/contracts/sf_contracts_extract.json
    Size : 10.8 KB

  tms_shipments_extract.json
    Path : /Volumes/supply_chain_dev/lp_analytics/raw_sources/tms/shipments/tms_shipments_extract.json
    Size : 222.2 KB

Total data size: 310.2 KB

Step 1 Complete — all source data ready!
